# IMDb Sentiment Analysis — Data Preparation
Load the IMDb dataset, inspect batches, build a vocabulary, and test text encoding and padding.

## 1. Imports and dataset

In [3]:
import sys
from pathlib import Path

root_dir = Path().resolve().parent
if str(root_dir) not in sys.path:
    sys.path.append(str(root_dir))

In [4]:
from torch.utils.data import DataLoader
from src.data.dataset import IMDbDataset
from src.data.tokenizer import *
from src.data.vocabulary import Vocabulary
from src.data.preprocessing import pad_or_truncate

In [5]:
train_dataset = IMDbDataset("../data/aclImdb", train=True)
test_dataset = IMDbDataset("../data/aclImdb", train=False)

print(len(train_dataset))  # 25000
print(len(test_dataset))   # 25000

review, label = train_dataset[0]
print(review)
print(label)

25000
25000
Bromwell High is a cartoon comedy. It ran at the same time as some other programs about school life, such as "Teachers". My 35 years in the teaching profession lead me to believe that Bromwell High's satire is much closer to reality than is "Teachers". The scramble to survive financially, the insightful students who can see right through their pathetic teachers' pomp, the pettiness of the whole situation, all remind me of the schools I knew and their students. When I saw the episode in which a student repeatedly tried to burn down the school, I immediately recalled ......... at .......... High. A classic line: INSPECTOR: I'm here to sack one of your teachers. STUDENT: Welcome to Bromwell High. I expect that many adults of my age think that Bromwell High is far fetched. What a pity that it isn't!
1


## 2. Batching with DataLoader
The default `DataLoader` collates labels into tensors, while raw review strings remain strings.

In [6]:
train_loader = DataLoader(
    train_dataset,
    batch_size=32,
    shuffle=True
)

test_loader = DataLoader(
    test_dataset,
    batch_size=32,
    shuffle=False
)

train_reviews, train_labels = next(iter(train_loader))
print(type(train_reviews))  # tuple of reviews
print(type(train_labels))   # torch.Tensor
print(len(train_reviews))   # 32
print(train_labels.shape)   # torch.Size([32])
print(train_reviews[0])

<class 'tuple'>
<class 'torch.Tensor'>
32
torch.Size([32])
Panic is a sneaky little gem of a film - you think you have it figured out by the first half hour only to realize, with great pleasure, that Henry Bromell is a much better writer/director than that. <br /><br />The film builds slowly, with one quietly devastating scene after another, all enacted perfectly by William H. Macy, Donald Sutherland, Neve Campbell, Tracey Ullman, John Ritter, and the most remarkable child actor I've seen in a long time, David Dorfman, as Macy's son, who delivers his lines as if they're completely unscripted thoughts being created in his mind. Rich and rewarding, this film will stay with you long after the credits have rolled.


## 3. Build the vocabulary
Build it using training reviews only, so information from the test set does not leak into preprocessing.

In [7]:
tokenizer = WhitespaceTokenizer()
vocab = Vocabulary(tokenizer, min_freq=2)
vocab.build([review for review, _ in train_dataset])  # Build vocabulary from training reviews

In [8]:
text = "this movie is great"
print(f"Original text: {text}")

encoded_text = vocab.encode(text)
print(f"Encoded text: {encoded_text}")

decoded_text = vocab.decode(encoded_text)
print(f"Decoded text: {decoded_text}")

Original text: this movie is great
Encoded text: [305, 383, 4, 1282]
Decoded text: this movie is great


## 4. Padding and truncation
Convert a review into token IDs first, then pad or truncate that sequence to the chosen length.

In [9]:
padded_sequence = pad_or_truncate(
    encoded_text,
    max_length=10,
    pad_id=vocab.PAD_ID
)
print(f"Processed sequence: {padded_sequence}")
print(f"Decoded padded sequence: {vocab.decode(padded_sequence)}")

Processed sequence: [305, 383, 4, 1282, 0, 0, 0, 0, 0, 0]
Decoded padded sequence: this movie is great <PAD> <PAD> <PAD> <PAD> <PAD> <PAD>


In [10]:
encoded_review = vocab.encode(train_reviews[0])
padded_review = pad_or_truncate(
    encoded_review,
    max_length=100,
    pad_id=vocab.PAD_ID
)

print(f"Padded sequence of first training review: {padded_review}")
print(f"Decoded padded sequence of first training review: {vocab.decode(padded_review)}")
print(f"Length of processed review: {len(padded_review)}")

Padded sequence of first training review: [62067, 4, 5, 15272, 1325, 8482, 54, 5, 306, 693, 156, 94, 156, 318, 99, 3133, 339, 219, 11, 1067, 1637, 1289, 437, 31, 19018, 190, 1282, 31569, 33, 4440, 62048, 4, 5, 35, 385, 9461, 38, 1426, 703, 153, 574, 306, 16901, 14667, 190, 85, 27594, 5052, 369, 220, 7085, 57, 62069, 2664, 219, 5661, 12120, 61960, 14413, 27824, 24871, 56454, 39593, 61986, 790, 53511, 62, 11, 391, 7401, 1000, 487, 1705, 1335, 26, 5, 1681, 2277, 2676, 62070, 14, 61970, 4789, 45, 9418, 207, 1644, 14, 146, 1760, 3141, 12411, 8627, 254, 3595, 26, 207, 4654, 5199, 62]
Decoded padded sequence of first training review: Panic is a sneaky little gem of a film - you think you have it figured out by the first half hour only to realize, with great pleasure, that Henry Bromell is a much better writer/director than that. <br /><br />The film builds slowly, with one quietly devastating scene after another, all enacted perfectly by William H. Macy, Donald Sutherland, Neve Campbell, Trac

## 5. Using DataLoader for batching
Using partial we incorporate some function parameters as a single callable for DataLoader's collate_fn parameter.


In [15]:
from functools import partial
from src.data.preprocessing import collate_fn

train_collate_fn = partial(collate_fn, vocab=vocab, max_length=512)

train_loader = DataLoader(
    train_dataset,
    batch_size=32,
    shuffle=True,
    collate_fn=train_collate_fn
)

In [16]:
inputs, labels = next(iter(train_loader))

print(inputs.shape)
print(labels.shape)
print(inputs.dtype)
print(labels.dtype)

torch.Size([32, 512])
torch.Size([32])
torch.int64
torch.int64
